# GloVe text classification

Run the cells in order.

## Headlines and topics
We will classify each headline as sport, business or science.

In [ ]:
import re, copy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(17)
torch.set_num_threads(1)
train = [('Team wins league title', 'sport'), ('Player scores winning goal', 'sport'), ('Coach announces new team', 'sport'), ('Club signs young player', 'sport'), ('Team prepares for final match', 'sport'), ('Fans celebrate cup victory', 'sport'), ('Bank raises interest rates', 'business'), ('Company reports record profit', 'business'), ('Shares fall after market losses', 'business'), ('Investors buy company shares', 'business'), ('New business creates jobs', 'business'), ('Bank announces lower profit', 'business'), ('Scientists discover new planet', 'science'), ('Researchers develop new medicine', 'science'), ('Space telescope finds distant star', 'science'), ('Scientists study ocean warming', 'science'), ('Research reveals new species', 'science'), ('Space mission reaches moon', 'science')]
valid = [('Club wins final match', 'sport'), ('Coach celebrates league victory', 'sport'), ('Company shares rise', 'business'), ('Investors expect record profit', 'business'), ('Researchers study distant planet', 'science'), ('Telescope discovers new star', 'science')]
labels = ["sport", "business", "science"]
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())
print(train[0])
print(tokenize(train[0][0]))

**Question:** What happens to capital letters when we tokenize the headline?

## GloVe embeddings
Each word is represented by a 100-dimensional vector, as we learned in our Word2Vec lecture. We load pretrained GloVe vectors into an embedding layer.

In [ ]:
import json
from pathlib import Path

glove = json.loads(Path("data/glove-headlines.json").read_text())
vocab = {"<PAD>": 0, "<UNK>": 1}
for text, _ in train:
    for word in tokenize(text):
        if word not in vocab:
            vocab[word] = len(vocab)
weights = torch.zeros(len(vocab), 100)
weights[1] = torch.randn(100) * 0.05
for word, index in vocab.items():
    if word in glove:
        weights[index] = torch.tensor(glove[word])
embedding = nn.Embedding.from_pretrained(weights, freeze=True, padding_idx=0)
def encode(text):
    return torch.tensor([vocab.get(w, 1) for w in tokenize(text)], dtype=torch.long)
ids = encode(train[0][0]).unsqueeze(0)
vectors = embedding(ids)
print("IDs:", ids.shape)
print("Vectors:", vectors.shape)
print("Train embedding?", embedding.weight.requires_grad)

**Question:** Change freeze=True to freeze=False. Which weights can training update?

## Combining word vectors
Averaging across words gives one 100-dimensional vector for the headline.

We can combine word vectors using different strategies: take the mean or the maximum in each dimension.

In [ ]:
features = vectors.mean(dim=1)
classifier = nn.Linear(100, 3)
scores = classifier(features)
print("Headline:", features.shape)
print("Scores:", scores.shape)
print("Untrained prediction:", labels[scores.argmax(dim=-1).item()])
reverse_vectors = embedding(ids.flip(dims=[1]))
print("Same average:", torch.allclose(features, reverse_vectors.mean(dim=1)))

**Question:** Replace vectors.mean(dim=1) with vectors.max(dim=1).values. Compare the resulting vectors.

## Padding
Headlines have different lengths. We add PAD values so they fit in one tensor. The mask selects the real words when we calculate the average.

In [ ]:
def make_batch(examples):
    ids = pad_sequence([encode(t) for t, _ in examples], batch_first=True, padding_value=0)
    targets = torch.tensor([labels.index(label) for _, label in examples])
    return ids, targets
batch_ids, targets = make_batch([train[0], train[1], train[4]])
mask = batch_ids.ne(0).unsqueeze(-1)
word_vectors = embedding(batch_ids)
features = (word_vectors * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
print("IDs:", batch_ids.shape)
print("Mask:", mask.shape)
print("Features:", features.shape)
print("Targets:", targets.shape)
assert torch.allclose(features[0], embedding(encode(train[0][0])).mean(dim=0))

**Question:** Calculate the average with and without the mask. What happens to the shorter headline?

## The classifier
A linear layer maps the 100-dimensional headline vector to three topic scores.

In [ ]:
class HeadlineClassifier(nn.Module):
    def __init__(self, weights, freeze=True):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weights.clone(), freeze=freeze, padding_idx=0)
        self.classifier = nn.Linear(weights.shape[1], len(labels))

    def forward(self, ids):
        mask = ids.ne(0).unsqueeze(-1)
        vectors = self.embedding(ids)
        pooled = (vectors * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled)

model = HeadlineClassifier(weights)
print("Logits:", model(batch_ids).shape)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

**Question:** Add a hidden layer and ReLU. How many trainable parameters does the model have now?

## Training
Cross-entropy compares the scores with the correct topic. Backpropagation computes gradients; the optimizer updates the trainable weights.

In [ ]:
train_ids, train_targets = make_batch(train)
valid_ids, valid_targets = make_batch(valid)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_history = []
for epoch in range(120):
    model.train()
    optimizer.zero_grad()
    scores = model(train_ids)
    loss = criterion(scores, train_targets)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
print("Loss:", round(loss_history[0], 3), "→", round(loss_history[-1], 3))
model.eval()
with torch.no_grad():
    predictions = model(valid_ids).argmax(dim=-1)
print("Validation correct:", (predictions == valid_targets).sum().item(), "/", len(valid))
loss_history

**Question:** Unfreeze the embeddings, create a fresh model and optimizer, and train again. Compare the predictions.

## Predictions
Read the predicted topic for each headline. Words outside the training vocabulary receive the UNK ID.

In [ ]:
model.eval()
with torch.no_grad():
    for (text, gold), prediction in zip(valid, model(valid_ids).argmax(-1)):
        unknown = [w for w in tokenize(text) if w not in vocab]
        print(text, "| gold:", gold, "| predicted:", labels[prediction.item()], "| UNK:", unknown)
    a = encode("team wins league title").unsqueeze(0)
    b = encode("title league wins team").unsqueeze(0)
    assert torch.allclose(model(a), model(b), atol=1e-6)
    print("Reordering preserves scores:", torch.allclose(model(a), model(b), atol=1e-6))

**Question:** Reverse the words in a headline. Compare all three scores.

## From classification to NER
For NER, we keep a vector for every word and predict a tag at every word position.

In [ ]:
word_classifier = nn.Linear(100, 9)
word_vectors = model.embedding(batch_ids)
word_scores = word_classifier(word_vectors)
predicted_tags = word_scores.argmax(dim=-1)
print("Word vectors:", word_vectors.shape)
print("Tag scores:", word_scores.shape)
print("Tag IDs:", predicted_tags.shape)
print("Padding positions:", batch_ids.eq(0))
# Tag IDs are untrained. Ignore padded targets when computing the NER loss.

**Question:** Which axis does argmax(dim=-1) remove? Which axes remain?